# CI Agent — Competitive Intelligence Analysis

In [1]:
import pathlib
import sys
import os

import anthropic
import pandas as pd
from IPython.display import display, Markdown

_here = pathlib.Path().resolve()
if (_here / 'clinical-trial-optimizer').exists():
    CT_ROOT = _here / 'clinical-trial-optimizer'
elif (_here / 'src').exists():
    CT_ROOT = _here
elif (_here.parent / 'src').exists():
    CT_ROOT = _here.parent
else:
    raise RuntimeError(f'Cannot locate clinical-trial-optimizer root from {_here}')

sys.path.insert(0, str(CT_ROOT / 'src'))

from agent_prompt_functions import get_ci_agent_prompt
from ct_api import fetch_competing_trials

print(f'CT_ROOT : {CT_ROOT}')

CT_ROOT : /Users/rajeevkulkarni/ml-explorations/clinical-trial-optimizer


In [2]:
print('API key set' if os.environ.get('ANTHROPIC_API_KEY') else 'WARNING: ANTHROPIC_API_KEY not set')

API key set


In [3]:
# Paste I/E criteria here
ie_criteria = """
Inclusion Criteria:
- Participant must have at least 1 measurable lesion, according to Response Evaluation Criteria in Solid Tumors (RECIST) version 1.1, that has not been previously irradiated
- Participant must have histologically or cytologically confirmed, locally advanced or metastatic, non-squamous non-small cell lung cancer (NSCLC), characterized at or after the time of locally advanced or metastatic disease diagnosis by either epidermal growth factor receptor (EGFR) Exon 19del or Exon 21 L858R mutation
- A participant with a history of brain metastases must have had all lesions treated as clinically indicated (that is, no current indication for further definitive local therapy). Any definitive local therapy to brain metastases must have been completed at least 14 days prior to randomization and the participant can be receiving no greater than10 milligrams (mg) prednisone or equivalent daily for the treatment of intracranial disease
- Participant must have Eastern Cooperative Oncology Group (ECOG) status of 0 or 1
- Any toxicities from prior systemic anticancer therapy must have resolved to National Cancer Institute Common Terminology Criteria for Adverse Events (NCI CTCAE) Version 5.0 Grade 1 or baseline level (except for alopecia [any grade], Grade <= 2 peripheral neuropathy, or Grade <= 2 hypothyroidism stable on hormone replacement)
- A participant of childbearing potential must have a negative serum pregnancy test at screening and within 72 hours of the first dose of study treatment and must agree to further serum or urine pregnancy tests during the study
- Participant must have progressed on or after osimertinib monotherapy as the most recent line of treatment. Osimertinib must have been administered as either the first-line treatment for locally advanced or metastatic disease or in the second- line setting after prior treatment with first- or second-generation EGFR tyrosine kinase inhibitor (TKI) as a monotherapy. Participants who received either neoadjuvant and/or adjuvant treatment of any type are eligible if progression to locally advanced or metastatic disease occurred at least 12 months after the last dose of such therapy and then the participant progressed on or after osimertinib in the locally advanced or metastatic setting. Treatment with osimertinib must be discontinued at least 8 days (4 half-lives) prior to randomization (that is last dose no later than Day -8)

Exclusion Criteria:
- Participant received radiotherapy for palliative treatment of NSCLC less than 14 days prior to randomization
- Participant with symptomatic or progressive brain metastases
- Participant has history of or current evidence of leptomeningeal disease, or participant has spinal cord compression not definitively treated with surgery or radiation
- Participant has known small cell transformation
- Participant has a medical history of interstitial lung disease (ILD), including drug-induced ILD or radiation pneumonitis
- Participant has a history of clinically significant cardiovascular disease including, but not limited to diagnosis of deep vein thrombosis or pulmonary embolism within 4 weeks prior to randomization; myocardial infarction; unstable angina; stroke; transient ischemic attack; coronary/peripheral artery bypass graft; or acute coronary syndrome. Participant has a significant genetic predisposition to venous thromboembolic events. Participant has a prior history of venous thromboembolic events and is not on appropriate therapeutic anticoagulation as per National Comprehensive Cancer Network or local guidelines"
"""

print(f'Length of I/E criteria: {len(ie_criteria)} characters')

Length of I/E criteria: 3585 characters


In [4]:
# ⚠️ HUMAN APPROVAL REQUIRED — this cell makes a live API call to ClinicalTrials.gov
competing_trials_md = fetch_competing_trials(ie_criteria)
print(competing_trials_md[:500])

Search terms extracted: condition='NSCLC EGFR', intr='osimertinib'
_Search terms used: condition='NSCLC EGFR', intr='osimertinib' · 20 trials retrieved from ClinicalTrials.gov API v2_

| NCT ID | Trial Name | Phase | Status | Sponsor | Intervention | Primary Endpoint | N | I/E Criteria |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| NCT02317016 | Study to Assess the Effect of AZD9291 on the Blood Levels of Rosuvastatin, in Patients With EGFRm+ Non-small Cell Lung Cancer | PHASE1 | COMPLETED | AstraZeneca | N/A | Assessment of Maximum Plasma Concentr


In [5]:
competing_trials_md

"_Search terms used: condition='NSCLC EGFR', intr='osimertinib' · 20 trials retrieved from ClinicalTrials.gov API v2_\n\n| NCT ID | Trial Name | Phase | Status | Sponsor | Intervention | Primary Endpoint | N | I/E Criteria |\n| --- | --- | --- | --- | --- | --- | --- | --- | --- |\n| NCT02317016 | Study to Assess the Effect of AZD9291 on the Blood Levels of Rosuvastatin, in Patients With EGFRm+ Non-small Cell Lung Cancer | PHASE1 | COMPLETED | AstraZeneca | N/A | Assessment of Maximum Plasma Concentration (Cmax) for Rosuvastatin After a Single Dose Alone and in Combination With AZD9291 | 44 | For inclusion in the study patients must fulfil the following criteria:  1. Male or female, aged at least 18 years. 2. Histological or cytological confirmation diagnosis of NSCLC. 3. Radiological documentation of disease progression while on a previous continuous treatment with an EGFR TKI, eg, gefitinib, erlotinib or afatinib. In addition, other lines of therapy may have been given. All patients 

In [6]:
ci_prompt = get_ci_agent_prompt(ie_criteria, competing_trials_md)
print(f'Prompt length: {len(ci_prompt)} chars')

Prompt length: 29966 chars


In [7]:
# ⚠️ HUMAN APPROVAL REQUIRED — this cell calls the Claude API
client = anthropic.Anthropic()

response = client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=4096,
    messages=[{'role': 'user', 'content': ci_prompt}],
)

display(Markdown(response.content[0].text))

# COMPETITIVE INTELLIGENCE ANALYSIS: OSIMERTINIB-PROGRESSION EGFR+ NSCLC TRIAL

---

## PRE-TABLE VERIFICATION: TREATMENT-RELATED CRITERIA FILTERING

**Total identified treatment-related I/E criteria: 5**

| Criterion | Type | Assessment |
|-----------|------|-----------|
| "Progressed on or after osimertinib monotherapy as most recent line" | Inclusion | Treatment-related: specifies prior osimertinib requirement and line of therapy |
| "Osimertinib administered as first-line OR second-line after first/second-gen EGFR TKI" | Inclusion | Treatment-related: specifies eligible prior treatment regimens and lines |
| "Participants who received neoadjuvant/adjuvant treatment eligible if progression ≥12 months after last dose, then progressed on/after osimertinib" | Inclusion | Treatment-related: specifies prior therapy washout and osimertinib progression requirement |
| "Osimertinib must be discontinued ≥8 days prior to randomization" | Inclusion | Treatment-related: specifies drug washout timing |
| "Received radiotherapy for palliative treatment of NSCLC <14 days prior to randomization" | Exclusion | Not treatment-related: radiotherapy is not systemic therapy; EXCLUDED from analysis |
| "History of clinically significant cardiovascular disease including DVT, PE within 4 weeks, MI, stroke, etc." | Exclusion | Not treatment-related: clinical condition, not systemic therapy; EXCLUDED from analysis |
| "History of interstitial lung disease" | Exclusion | Not treatment-related: clinical condition, not systemic therapy; EXCLUDED from analysis |
| "History of or current evidence of leptomeningeal disease or spinal cord compression" | Exclusion | Not treatment-related: anatomical disease state, not systemic therapy; EXCLUDED from analysis |
| "Small cell transformation" | Exclusion | Not treatment-related: disease characteristic, not systemic therapy; EXCLUDED from analysis |

**Treatment-Related Criteria Count:**
- **Total Aligned criteria: 2**
- **Total More Restrictive criteria: 3**
- **Total Less Restrictive criteria: 0**
- **Total White Space criteria: 0**
- **Grand Total: 5**

---

## COMPETITIVE INTELLIGENCE OUTPUT TABLE

| Type | Original Criterion | Gap Type | Queriable | Competing Trial | Competitive Position | Explanation | Recommendation |
|------|-------------------|----------|-----------|-----------------|----------------------|-------------|-----------------|
| Inclusion | "Participant must have progressed on or after osimertinib monotherapy as the most recent line of treatment" | More Restrictive | Yes | NCT07100080 (IZABRIGHT-Lung01), NCT06417814 (Dato-DXd study) | More Restrictive | **NCT07100080** and **NCT06417814** both require "progressive disease on 3rd-generation EGFR-TKI (osimertinib, furmonertinib, lazertinib) as most recent line," but allow COMBINATION therapy (e.g., TKI + chemotherapy or TKI + bevacizumab), whereas this trial explicitly requires osimertinib MONOTHERAPY. This narrows the eligible population to patients who specifically failed osimertinib alone, excluding patients who progressed on osimertinib-containing combinations. | Consider broadening to: "Include participants who have progressed on osimertinib, administered as monotherapy or in combination, as the most recent line of treatment for locally advanced or metastatic NSCLC." Alternatively, add explicit cohort: "OR progressed on osimertinib-based combination therapy [specify regimen]." |
| Inclusion | "Osimertinib must have been administered as either first-line treatment for locally advanced or metastatic disease OR in second-line setting after first- or second-generation EGFR TKI monotherapy" | Aligned | Yes | NCT06838273 (BL-B01D1 study), NCT05382728 (TY-9591/FLETEO), NCT04181060 (Osimertinib ± Bevacizumab study) | Aligned | **NCT06838273**, **NCT05382728**, and **NCT04181060** all specify osimertinib or other 3rd-gen TKI as first-line therapy for locally advanced or metastatic EGFR-mutant NSCLC. This trial's requirement for first-line osimertinib OR second-line after 1st/2nd-gen TKI aligns with the standard sequential EGFR-TKI progression observed across the field. | Already captured by existing I/E criteria. |
| Inclusion | "Participants who received neoadjuvant and/or adjuvant treatment of any type are eligible if progression to locally advanced or metastatic disease occurred ≥12 months after last dose, then progressed on or after osimertinib in the metastatic setting" | More Restrictive | Yes | NCT05120349 (Osimertinib adjuvant Stage IA2-IA3 study) | More Restrictive | **NCT05120349** enrolls patients with early-stage NSCLC (Stage IA2-IA3) post-surgical resection receiving adjuvant osimertinib, whereas this trial requires patients to have progressed to locally advanced or metastatic disease at least 12 months after adjuvant/neoadjuvant therapy and then progressed on osimertinib. This trial's washout requirement (≥12 months from last adjuvant dose to progression to metastatic disease, then progression on osimertinib) is more restrictive: it requires two sequential progression events (metastatic progression + osimertinib progression), whereas **NCT05120349** includes early-stage patients before metastatic progression occurs. | Consider clarifying eligibility for the adjuvant-treated population: "Include participants who received neoadjuvant or adjuvant systemic therapy for Stage I-III NSCLC and subsequently progressed to locally advanced or metastatic disease ≥12 months after completion of adjuvant/neoadjuvant therapy, provided they have since progressed on osimertinib monotherapy." This maintains the dual-progression requirement while clarifying the timeline. |
| Inclusion | "Treatment with osimertinib must be discontinued ≥8 days (4 half-lives) prior to randomization" | Aligned | Yes | NCT06417814 (Dato-DXd study), NCT07100080 (IZABRIGHT-Lung01) | Aligned | **NCT06417814** requires "≤14 days from last osimertinib dose to randomization" and **NCT07100080** similarly specifies prior osimertinib discontinuation prior to enrollment. The 8-day washout (4 half-lives of osimertinib) is pharmacologically standard and aligned with competing trials' requirements. | Already captured by existing I/E criteria. |
| Inclusion | "Participant must have progressed on or after osimertinib monotherapy... osimertinib administered as first-line treatment for locally advanced or metastatic disease OR second-line after prior 1st/2nd-gen EGFR TKI" (combined line specification) | More Restrictive | Yes | NCT07100080 (IZABRIGHT-Lung01), NCT06417814 (Dato-DXd study), NCT05382728 (TY-9591/FLETEO) | More Restrictive | **NCT07100080** and **NCT06417814** both allow prior osimertinib in adjuvant, locally advanced, OR metastatic settings without explicit line restriction. **NCT05382728** (TY-9591/FLETEO) specifies "no prior systemic antitumor therapy for locally advanced or metastatic NSCLC" (treatment-naive first-line only). This trial restricts osimertinib to first-line OR second-line (after 1st/2nd-gen TKI only), excluding patients who received osimertinib in third-line or later metastatic settings (e.g., after chemotherapy following 1st-gen TKI failure). This is more restrictive than **NCT07100080** and **NCT06417814**, which allow any prior osimertinib in any line. | Consider broadening to align with field: "Include participants who have progressed on osimertinib monotherapy administered in any line of systemic therapy (first-line, second-line, or later) for locally advanced or metastatic NSCLC, or as adjuvant/neoadjuvant therapy." Rationale: **NCT07100080** and **NCT06417814** demonstrate that post-osimertinib patients from multiple treatment lines are appropriate for Phase II/III enrollment. |

---

## UPDATED TREATMENT-RELATED CRITERIA (COMPETITIVE INTELLIGENCE)

### **Based on Original Criterion:** 
*"Participant must have progressed on or after osimertinib monotherapy as the most recent line of treatment. Osimertinib must have been administered as either the first-line treatment for locally advanced or metastatic disease or in the second-line setting after prior treatment with first- or second-generation EGFR tyrosine kinase inhibitor (TKI) as a monotherapy."*

- **Suggested Addition (Gap Type: More Restrictive, vs NCT07100080, NCT06417814):**  
  "Clarify that participants may have received osimertinib-containing combination therapy (e.g., osimertinib + bevacizumab, osimertinib + chemotherapy) in prior lines, provided the most recent line of treatment for locally advanced or metastatic disease was osimertinib monotherapy and the participant has documented progression on or after osimertinib."

**Updated Combined Criterion:**  
"Participant must have progressed on or after osimertinib monotherapy as the most recent line of treatment for locally advanced or metastatic disease. Osimertinib must have been administered as either first-line treatment for locally advanced or metastatic disease OR in the second-line setting after prior first- or second-generation EGFR TKI monotherapy. Prior osimertinib-containing combination therapy in earlier lines is permitted."

---

### **Based on Original Criterion:**  
*"Participants who received either neoadjuvant and/or adjuvant treatment of any type are eligible if progression to locally advanced or metastatic disease occurred at least 12 months after the last dose of such therapy and then the participant progressed on or after osimertinib in the locally advanced or metastatic setting."*

- **Suggested Addition (Gap Type: More Restrictive, vs NCT05120349):**  
  "Explicitly define 'progression to locally advanced or metastatic disease' as the first event triggering the 12-month clock, and clarify the sequential requirement: (1) adjuvant/neoadjuvant therapy completion → (2) ≥12 months observation or disease-free interval → (3) progression to metastatic disease → (4) treatment with osimertinib → (5) progression on or after osimertinib. This dual-progression structure is justified by the trial's post-osimertinib focus but is more restrictive than early-stage adjuvant trials like **NCT05120349**."

**Updated Combined Criterion:**  
"Participants who received neoadjuvant and/or adjuvant systemic therapy for Stage I-III NSCLC are eligible if: (a) ≥12 months have elapsed between completion of adjuvant/neoadjuvant therapy and documented progression to locally advanced or metastatic disease, AND (b) the participant has subsequently progressed on or after osimertinib monotherapy in the metastatic setting."

---

## COMPETITIVE INTELLIGENCE SUMMARY

### **Disease & Biomarker Context**
This trial targets **locally advanced or metastatic non-squamous NSCLC with EGFR exon 19 deletion or exon 21 L858R mutation**, in patients who have progressed on **osimertinib monotherapy**. The population is narrowly defined as post-osimertinib-progression EGFR-mutant NSCLC with specific biomarker requirements.

### **Competing Trial Landscape**
- **20 competing trials** identified; 15 have sufficient I/E data for analysis.
- **Phases:** Phase I (4), Phase II (5), Phase III (5), observational/retrospective (2).
- **Top Sponsors:** AstraZeneca (5 trials), NCI (1), industry sponsors (BMS, Sichuan Baili, TYK Medicines, others).
- **Field Status:** CROWDED. Osimertinib-focused EGFR+ NSCLC space is heavily populated with:
  - First-line osimertinib trials (±combinations): **NCT04181060, NCT06838273, NCT05382728, NCT07295821, NCT07028489**
  - Post-osimertinib progression trials: **NCT07100080 (IZABRIGHT-Lung01), NCT06417814 (Dato-DXd)**
  - Biomarker/resistance studies: **NCT03969823, NCT03239340**
  - Adjuvant osimertinib: **NCT05120349**

### **Competitive Positioning Overview**
This trial is **MODERATELY MORE RESTRICTIVE** than the competing landscape in the post-osimertinib space:

1. **Osimertinib Monotherapy Requirement (Most Restrictive):**  
   - This trial explicitly requires osimertinib **monotherapy** as the most recent line.
   - **NCT07100080** and **NCT06417814** allow patients who progressed on osimertinib-**containing combinations** (e.g., osimertinib + bevacizumab, osimertinib + chemotherapy).
   - **Enrollment Impact:** MAJOR. Excluding osimertinib-combination-failure patients reduces eligible population size, especially in regions/centers where osimertinib + bevacizumab or osimertinib + chemotherapy are standard practice.

2. **Line-of-Therapy Restriction (More Restrictive):**  
   - This trial restricts osimertinib to **first-line or second-line** (after 1st/2nd-gen TKI only).
   - **NCT07100080** and **NCT06417814** allow osimertinib progression "in adjuvant, locally advanced, or metastatic setting" without explicit line number caps. This implicitly permits third-line or later osimertinib use (e.g., after chemotherapy following first-gen TKI → second-gen TKI → chemotherapy → osimertinib).
   - **Enrollment Impact:** MINOR-to-MODERATE. Most treatment sequences follow the conventional first-gen TKI → osimertinib path, so third-line+ osimertinib is less common. However, patients with complex treatment histories (e.g., neoadjuvant → adjuvant → metastatic progression → osimertinib) may be excluded if osimertinib is applied in a later-numbered line.

3. **Adjuvant/Neoadjuvant Washout (More Restrictive):**  
   - This trial requires ≥12 months from adjuvant therapy completion to metastatic progression, PLUS subsequent osimertinib progression.
   - **NCT05120349** (adjuvant osimertinib in early-stage) does not have this dual-progression requirement; it enrolls patients upfront post-resection.
   - **NCT07100080** and **NCT06417814** accept patients with adjuvant osimertinib history but do not explicitly mandate the 12-month off-treatment interval before metastatic progression.
   - **Enrollment Impact:** MAJOR. This narrows the adjuvant-treated population to those who relapse after a disease-free interval AND then progress on osimertinib—a smaller subset than simple "prior adjuvant therapy" cohorts.

4. **Osimertinib Washout (8 days) (Aligned):**  
   - Standard across **NCT06417814, NCT07100080**; not a competitive disadvantage.

### **White Space Opportunities**
- **No white space identified** in treatment-line or osimertinib-prior-exposure criteria. The competing landscape (especially **NCT07100080, NCT06417814**) comprehensively covers post-osimertinib populations.
- **Potential differentiation:** If the trial explicitly **broadens** to allow osimertinib-combination failures, it would capture an underserved population (patients on osimertinib + bevacizumab or osimertinib + chemotherapy combinations who progress). This is NOT currently addressed by most competing Phase III trials and represents a **white space opportunity** for enrollment expansion.

### **Primary Recommendation**

**Broaden the osimertinib treatment criterion to align with Phase III competitors and capture osimert